# Collaborative baseline: Matrix Factorization

Biased matrix factorization pada rating eksplisit. Model dilatih hanya pada train; rekomendasi dibatasi ke warm candidates dan item yang sudah dilihat di train dikeluarkan.

In [1]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from data_loader import load_data, relevant_items_val
from matrix_factorization import MatrixFactorizationRecommender

train, val, cand_set, seen = load_data(str(ROOT / 'data') + '/')
relevant = relevant_items_val(val, cand_set, seen, min_rating=4)
print(f'train={len(train):,}; validation={len(val):,}; candidates={len(cand_set):,}; evaluable users={len(relevant):,}')

train=427,957; validation=41,284; candidates=22,753; evaluable users=11,850


## Fit model

Rating digunakan sebagai explicit feedback; nilai prediksi mengurutkan item.

In [2]:
model = MatrixFactorizationRecommender(
    n_factors=32, n_epochs=10, learning_rate=0.01,
    regularization=0.02, min_rating=4, random_state=42,
).fit(train, cand_set)
print(f'Fit complete: {len(model.user_index):,} users, {len(model.item_index):,} items, {model.n_factors} factors')

Fit complete: 54,483 users, 22,753 items, 32 factors


## Evaluate ranking

Recall@10 dan NDCG@10 dihitung pada user dengan setidaknya satu item relevan di validation. Relevansi mengikuti loader: rating ≥ 4, item termasuk warm candidates, dan user muncul di train.

In [3]:
def evaluate_at_k(model, relevant, seen, k=10):
    recalls, ndcgs = [], []
    for user_id, relevant_items in relevant.items():
        recs = model.recommend(user_id, seen, k=k)
        hits = [item in relevant_items for item in recs]
        recalls.append(sum(hits) / len(relevant_items))
        dcg = sum(hit / np.log2(rank + 2) for rank, hit in enumerate(hits))
        ideal_hits = min(len(relevant_items), k)
        idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_hits))
        ndcgs.append(dcg / idcg if idcg else 0.0)
    return {'Recall@10': float(np.mean(recalls)), 'NDCG@10': float(np.mean(ndcgs))}

metrics = evaluate_at_k(model, relevant, seen, k=10)
metrics

{'Recall@10': 0.0008711182550844998, 'NDCG@10': 0.00044623202846490634}

## Inspect recommendations

Contoh user diambil dari validation yang eligible.

In [4]:
example_user = next(iter(relevant))
print('user:', example_user)
print('recommendations:', model.recommend(example_user, seen, k=10))
print('relevant validation items:', relevant[example_user])

user: AE22B7BMXT7GAGY3XLHICMESNAPA
recommendations: ['B003JCF5Y2', 'B09LTZ1QXP', 'B074P6W2G1', 'B000068O44', 'B00008XETI', 'B06VX78R9Z', 'B00IPHITHG', 'B00FM0MSDK', 'B08129P2XY', 'B0009FJ1NS']
relevant validation items: {'B0BYNM15YB'}
